In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment and packages for this notebook.
'''
conda create -n ls-export python=3.11 -y
conda activate ls-export

pip install \
  jupyterlab ipykernel \
  label-studio-sdk \
  label-studio-converter \
  opencv-python-headless \
  numpy \
  matplotlib \
  requests


python -m ipykernel install --user --name ls-export --display-name "ls-export"

In [ ]:
# === Cell 3: Imports ===

import os
import csv
import json
import random
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import numpy as np
import cv2
from label_studio_sdk import LabelStudio
from label_studio_converter.brush import decode_rle

import matplotlib.pyplot as plt  # for optional visual checks

In [ ]:
# === Cell 4: Config ===

# --- Label Studio server configuration ---

# Base URL of the Label Studio instance
LS_URL = "https://jupyter.ki-werkstatt.htw-aalen.de/labelstudio/fio.h/"  
LS_API_KEY = os.environ.get("LS_API_KEY", "")  # read from env var; alternatively use the input() prompt below
#LS_API_KEY = input("Enter Label Studio API key (not stored on disk): ").strip()
LS_PROJECT_ID = 5  


# --- Filesystem configuration ---

# Source images and export output directory.
IMAGE_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Initial_Image_Dataset/1080_resolution")
OUTPUT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260513_Initial_Export_v2")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# --- YOLO & splitting configuration ---

# YOLO class mapping: LS brush label name -> YOLO class id
CLASS_MAP = {
    "body": 0,
    "cap": 1,
    "label": 2,
}

MATERIAL_CLASSES = ["HDPE", "PET", "PP", "PS"]

# Objects required per material, and how many are held out as test objects.
REQUIRED_OBJECTS_PER_MATERIAL = 10
NUM_TEST_OBJECTS_PER_MATERIAL = 2  # chosen randomly per material
RANDOM_SEED = 42  # seed for all random decisions


# --- Polygon extraction / export parameters ---

POLYGON_MIN_AREA = 100  # minimum contour area (px) kept as a polygon
POLYGON_EPSILON_REL = 0.001  # approxPolyDP epsilon as a fraction of contour perimeter



print("Configuration:")
print(f"  LS_URL        = {LS_URL}")
print(f"  LS_PROJECT_ID = {LS_PROJECT_ID}")
print(f"  IMAGE_ROOT    = {IMAGE_ROOT}")
print(f"  OUTPUT_ROOT   = {OUTPUT_ROOT.resolve()}")
print(f"  CLASS_MAP     = {CLASS_MAP}")
print(f"  RANDOM_SEED   = {RANDOM_SEED}")
print(f"  MATERIALS     = {MATERIAL_CLASSES}")
print(f"  POLY_MIN_AREA = {POLYGON_MIN_AREA}")
print(f"  POLY_EPS_REL  = {POLYGON_EPSILON_REL}")

In [ ]:
# === Cell 5: LS Client and Task Export ===

def get_ls_client(base_url: str = LS_URL, api_key: str = LS_API_KEY) -> LabelStudio:
    """Create a Label Studio client."""
    if not api_key:
        raise RuntimeError("LS_API_KEY is empty. Re-run the config cell and provide a key.")
    client = LabelStudio(base_url=base_url, api_key=api_key)
    return client


def export_tasks(client: LabelStudio, project_id: int = LS_PROJECT_ID):
    """Export all tasks with annotations in raw JSON format using the v2 SDK."""
    print(f"Exporting tasks from project {project_id} at {LS_URL} ...")
    tasks = client.projects.exports.as_json(project_id)
    print(f"Retrieved {len(tasks)} tasks.")
    return tasks


def detect_image_key(tasks):
    """Infer which key in task['data'] holds the image path/URL."""
    if not tasks:
        raise RuntimeError("No tasks found in export.")

    sample = tasks[0]
    data = sample.get("data", {})
    img_key = None
    for k, v in data.items():
        if isinstance(v, str):
            lower = v.lower()
            if lower.endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")):
                img_key = k
                break

    if not img_key:
        img_key = "image"
        print("WARNING: Could not auto-detect image key; falling back to 'image'. "
              "Check your Label Studio task JSON if this fails.")
    else:
        print(f"Detected image data key: '{img_key}'")
    return img_key


def resolve_image_path(image_value):
    """
    Convert task['data'][img_key] to a local Path under IMAGE_ROOT.

    Handles LS local-files URLs (/data/local-files/?d=subdir/image1.png), http(s)
    URLs and plain paths. Only the filename is used; images are expected directly
    under IMAGE_ROOT.
    """
    # Case 1: LS local-files URL or /data/local-files path
    if isinstance(image_value, str) and ("local-files" in image_value or image_value.startswith("/data/local-files")):
        parsed = urlparse(image_value)
        qs = parse_qs(parsed.query)
        d_vals = qs.get("d") or qs.get("d[]")
        if d_vals:
            rel_path = d_vals[0]
            filename = Path(rel_path).name
            return IMAGE_ROOT / filename
        else:
            filename = Path(parsed.path).name
            return IMAGE_ROOT / filename

    # Case 2: plain http/https URL – use basename
    parsed = urlparse(image_value)
    if parsed.scheme in ("http", "https"):
        filename = Path(parsed.path).name
        return IMAGE_ROOT / filename

    # Case 3: plain path string
    p = Path(image_value)
    if p.is_absolute():
        if p.exists():
            return p
        return IMAGE_ROOT / p.name

    return IMAGE_ROOT / p

In [ ]:
# === Cell 6: Mask to YOLO Conversion ===

def brush_result_to_mask(result):
    """
    Convert a Label Studio BrushLabels result into a binary mask (H, W) uint8.
    Uses decode_rle from label-studio-converter.
    """
    rle = result["value"]["rle"]
    h = result["original_height"]
    w = result["original_width"]

    rgba = decode_rle(rle).reshape((h, w, 4))
    mask = (rgba[:, :, 3] > 0).astype("uint8")  # alpha channel
    return mask


def mask_to_polygons(
    mask,
    min_area: int = POLYGON_MIN_AREA,
    epsilon_rel: float = POLYGON_EPSILON_REL,
):
    """
    Extract external contours as polygons, skipping very small areas.

    Returns:
      polys: list of approximated polygons (each as np.ndarray of shape (N,1,2))
      skipped_small: number of contours skipped because area < min_area
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    skipped_small = 0

    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            skipped_small += 1
            continue
        epsilon = epsilon_rel * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        polys.append(approx)

    return polys, skipped_small


def build_region_metadata_map(results):
    """
    Build a map from region result id -> {'type': str | None, 'color': [str]}
    using perRegion Choices results.

    Assumes labeling config defines:
      - Choices name="type_body" / "type_label" / "type_cap"
      - Choices name="color"
    and that perRegion results share the same 'id' as the region result.
    """
    meta_map = {}
    for r in results:
        if r.get("type") != "choices":
            continue
        r_id = r.get("id")
        if not r_id:
            continue

        from_name = r.get("from_name")
        value = r.get("value") or {}
        choices = value.get("choices") or []
        if not choices:
            continue

        entry = meta_map.setdefault(r_id, {"type": None, "color": []})

        if from_name in {"type_body", "type_label", "type_cap"}:
            entry["type"] = choices[0]
        elif from_name == "color":
            entry["color"] = choices

    return meta_map


def write_yolo_seg_file(
    label_path: Path,
    w: int,
    h: int,
    results,
    metadata_rows: list,
    image_path: Path,
    class_map: dict,
):
    """
    Create one YOLOv8 segmentation .txt file from LS results for a single image.
    Also appends per-region metadata rows to metadata_rows.

    Returns:
      ok (bool): True if at least one polygon was written, False otherwise
      skipped_small (int): number of polygons skipped due to area < POLYGON_MIN_AREA
    """
    lines = []
    region_meta = build_region_metadata_map(results)
    total_skipped_small = 0

    for res in results:
        if res.get("type") != "brushlabels":
            continue

        brushlabels = res["value"].get("brushlabels")
        if not brushlabels:
            continue

        label_name = brushlabels[0]
        if label_name not in class_map:
            print(f"WARNING: label '{label_name}' not in CLASS_MAP, skipping.")
            continue

        cls_id = class_map[label_name]

        mask = brush_result_to_mask(res)
        polys, skipped_small = mask_to_polygons(mask)
        total_skipped_small += skipped_small

        had_poly = False
        for poly in polys:
            xs = poly[:, 0, 0].astype(float) / float(w)
            ys = poly[:, 0, 1].astype(float) / float(h)
            coords = np.stack([xs, ys], axis=1).reshape(-1)
            if len(coords) < 6:
                continue
            coord_str = " ".join(f"{c:.6f}" for c in coords)
            lines.append(f"{cls_id} {coord_str}")
            had_poly = True

        if had_poly:
            region_id = res.get("id")
            meta = region_meta.get(region_id, {})
            meta_type = meta.get("type")
            meta_color_list = meta.get("color") or []
            meta_color_str = ",".join(meta_color_list)

            metadata_rows.append({
                "image": image_path.name,
                "region_id": region_id,
                "label": label_name,
                "class_id": cls_id,
                "type": meta_type,
                "color": meta_color_str,
            })

    if not lines:
        return False, total_skipped_small

    label_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return True, total_skipped_small

In [ ]:
# === Cell 7: Build Flat YOLO Dataset ===

def build_yolo_dataset_flat(
    tasks,
    image_root: Path = IMAGE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    class_map: dict = CLASS_MAP,
):
    """
    Convert LS tasks to YOLO segmentation labels and create a flat dataset:

      images/all  - all successfully converted images
      labels/all  - corresponding YOLO label files

    Returns:
      processed_images: list of dicts with keys:
        - image_path (Path)
        - image_name (str)
      metadata_rows: list of region-level metadata dicts.
      total_skipped_small: total number of polygons skipped due to area < POLYGON_MIN_AREA
    """
    img_key = detect_image_key(tasks)

    processed_images = []
    metadata_rows = []
    total_skipped_small = 0

    img_all_dir = output_root / "images" / "all"
    lbl_all_dir = output_root / "labels" / "all"
    img_all_dir.mkdir(parents=True, exist_ok=True)
    lbl_all_dir.mkdir(parents=True, exist_ok=True)

    for task in tasks:
        data = task.get("data", {})
        if img_key not in data:
            continue

        image_value = data[img_key]
        image_path = resolve_image_path(image_value)

        if not image_path.exists():
            print(f"WARNING: image not found on disk: {image_path}")
            continue

        base_name = image_path.stem
        annotations = task.get("annotations") or []
        if not annotations:
            continue

        # Exactly one annotation per image is expected.
        if len(annotations) > 1:
            raise RuntimeError(
                f"Task for image {image_path} has {len(annotations)} annotations; "
                "this notebook assumes exactly one annotation per image."
            )

        ann = annotations[0]
        results = ann.get("result") or []
        if not results:
            continue

        # Width/height: from brush result if present, else from image
        w = h = None
        for r in results:
            if "original_width" in r and "original_height" in r:
                w = r["original_width"]
                h = r["original_height"]
                break
        if w is None or h is None:
            img = cv2.imread(str(image_path))
            if img is None:
                print(f"WARNING: could not read image to get size: {image_path}")
                continue
            h, w = img.shape[:2]

        label_path = lbl_all_dir / f"{base_name}.txt"
        ok, skipped_small = write_yolo_seg_file(
            label_path, w, h, results, metadata_rows, image_path, class_map
        )
        total_skipped_small += skipped_small

        if ok:
            # Copy image into images/all
            dst_img = img_all_dir / image_path.name
            if not dst_img.exists():
                dst_img.write_bytes(image_path.read_bytes())

            processed_images.append({
                "image_path": dst_img,
                "image_name": dst_img.name,
            })
        else:
            if label_path.exists():
                label_path.unlink()

    print(f"Converted {len(processed_images)} images with polygons to YOLO labels.")

    # Minimal data.yaml; train/val splits are set by the training notebook.
    data_yaml = output_root / "data.yaml"
    class_names = [None] * (max(class_map.values()) + 1)
    for name, idx in class_map.items():
        class_names[idx] = name

    yaml_lines = [
        f"path: {output_root.resolve()}",
        "train: images/all",
        "val: images/all",
        "",
        "names:",
    ] + [f"  {i}: {n}" for i, n in enumerate(class_names)]
    data_yaml.write_text("\n".join(yaml_lines), encoding="utf-8")
    print(f"Wrote YOLO data.yaml to {data_yaml}")

    # Region-level metadata -> metadata_raw.csv
    if metadata_rows:
        metadata_csv = output_root / "metadata_raw.csv"
        with metadata_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=["image", "region_id", "label", "class_id", "type", "color"],
            )
            writer.writeheader()
            writer.writerows(metadata_rows)
        print(f"Wrote metadata for {len(metadata_rows)} regions to {metadata_csv}")
    else:
        print("No metadata rows collected (no polygons exported or no metadata present).")

    return processed_images, metadata_rows, total_skipped_small

In [ ]:
# === Cell 8: Object Index and Split ===

def parse_material_object_perspective(filename: str):
    """
    Parse material, object_id, perspective_id from a filename like:
      HDPE_obj_01_per_4_2026-04-21.png

    Returns (material, object_id, perspective_id); raises if the pattern does not match.
    """
    stem = Path(filename).stem
    parts = stem.split("_")
    # Expected minimal pattern: MATERIAL_obj_<id>_per_<perspective>_...
    if len(parts) < 5:
        raise RuntimeError(
            f"Filename '{filename}' does not match expected pattern 'MATERIAL_obj_ID_per_PERSPECTIVE_...'. Parts: {parts}"
        )

    material = parts[0]
    if parts[1] != "obj" or parts[3] != "per":
        raise RuntimeError(
            f"Filename '{filename}' does not match '..._obj_..._per_...' pattern. Parts: {parts}"
        )

    object_id = parts[2]
    perspective_id = parts[4]

    return material, object_id, perspective_id


def build_object_index(processed_images, metadata_rows):
    """
    Build object-level index from exported images (processed_images)
    and region-level metadata_rows.

    Returns:
      objects: dict[(material, object_id)] -> dict with:
        - material
        - object_id
        - image_names: set of image filenames
        - perspectives: set of perspective IDs (strings)
        - num_regions: total number of regions
        - per_perspective_metadata: dict[perspective_id] -> set of (label, type, color) tuples
    """
    objects = {}

    # Pass 1: images / perspectives
    for rec in processed_images:
        img_name = rec["image_name"]
        material, object_id, perspective_id = parse_material_object_perspective(img_name)

        key = (material, object_id)
        obj = objects.setdefault(key, {
            "material": material,
            "object_id": object_id,
            "image_names": set(),
            "perspectives": set(),
            "num_regions": 0,
            "per_perspective_metadata": {},  # perspective_id -> set of (label, type, color)
        })

        obj["image_names"].add(img_name)
        obj["perspectives"].add(perspective_id)
        obj["per_perspective_metadata"].setdefault(perspective_id, set())

    # Pass 2: region metadata
    for row in metadata_rows:
        img_name = row["image"]
        material, object_id, perspective_id = parse_material_object_perspective(img_name)
        key = (material, object_id)
        if key not in objects:
            # Skip metadata for images not in processed_images.
            continue

        obj = objects[key]
        obj["num_regions"] += 1

        label = row.get("label")
        type_val = row.get("type")
        color_val = row.get("color")
        meta_tuple = (label, type_val, color_val)
        obj["per_perspective_metadata"].setdefault(perspective_id, set()).add(meta_tuple)

    return objects


def compute_metadata_conflict(per_persp_meta: dict) -> int:
    """
    Compute a 0/1 metadata conflict flag for one object, per class label.

      Type: each region has one type; all regions of a label must share the same
        non-empty type.
      Color: regions may have several colors; all regions of a label must have the
        same color set (order ignored, an empty set counts as a distinct set).

    Returns:
      1 if any label violates these rules, otherwise 0.
    """
    if not per_persp_meta:
        return 0

    # Flatten all metadata tuples across perspectives into a single set
    all_sets = list(per_persp_meta.values())
    union_all = set().union(*all_sets) if all_sets else set()

    # Group type values and color sets by label ('body', 'label', 'cap', ...)
    type_values_by_label = {}
    color_sets_by_label = {}

    for (label, type_val, color_val) in union_all:
        if label is None:
            continue

        # ---- TYPE aggregation ----
        normalized_type = (type_val or "").strip()
        if normalized_type:
            type_values_by_label.setdefault(label, set()).add(normalized_type)

        # ---- COLOR aggregation ----
        # Normalize color_val into a set of tokens (possibly empty).
        if color_val is None:
            tokens = []
        else:
            tokens = [tok.strip() for tok in color_val.split(",") if tok.strip()]
        color_set = frozenset(tokens)
        color_sets_by_label.setdefault(label, set()).add(color_set)

    # Type rule: at most one distinct non-empty type per label
    for label, type_vals in type_values_by_label.items():
        if len(type_vals) > 1:
            return 1

    # Color rule: exactly one distinct color-set per label
    for label, color_sets in color_sets_by_label.items():
        if len(color_sets) > 1:
            return 1

    return 0


def build_object_row(obj, split: str, split_reason: str):
    """
    Build one object_split.csv row (counts and conflict flag) from an object entry.
    """
    material = obj["material"]
    object_id = obj["object_id"]
    image_names = obj["image_names"]
    num_regions = obj["num_regions"]
    per_persp_meta = obj["per_perspective_metadata"]

    metadata_conflict = compute_metadata_conflict(per_persp_meta)

    row = {
        "material": material,
        "object_id": object_id,
        "split": split,
        "split_reason": split_reason,
        "num_images": len(image_names),
        "num_regions": num_regions,
        "metadata_conflict": metadata_conflict,
    }
    return row


def aggregate_object_metadata_and_split(
    objects,
    output_root: Path = OUTPUT_ROOT,
    materials=MATERIAL_CLASSES,
    required_objects_per_material: int = REQUIRED_OBJECTS_PER_MATERIAL,
    num_test_objects_per_material: int = NUM_TEST_OBJECTS_PER_MATERIAL,
    seed: int = RANDOM_SEED,
):
    """
    Assign each object to a split (cv/test/skipped) and write object_split.csv.

    Writes object_split.csv with columns:
      - material
      - object_id
      - split: 'cv', 'test', or 'skipped'
      - split_reason: text reason for skipped/usage
      - num_images
      - num_regions
      - metadata_conflict: 0 or 1

    metadata_conflict: see compute_metadata_conflict.
    """
    rnd = random.Random(seed)

    # Group objects by material
    objs_by_material = {m: [] for m in materials}
    for (material, object_id), obj in objects.items():
        if material not in objs_by_material:
            objs_by_material.setdefault(material, []).append(obj)
        else:
            objs_by_material[material].append(obj)

    splits = []

    for material, obj_list in objs_by_material.items():
        if not obj_list:
            print(f"[INFO] Material {material}: no objects found.")
            continue

        obj_list_sorted = sorted(obj_list, key=lambda o: o["object_id"])
        num_objects = len(obj_list_sorted)
        print(f"[INFO] Material {material}: found {num_objects} objects.")

        if num_objects < required_objects_per_material:
            reason = (f"insufficient_objects_for_material: "
                      f"found {num_objects}, required >= {required_objects_per_material}")
            print(f"[WARNING] {reason} for material {material}. Skipping this material in cv/test assignment.")
            for obj in obj_list_sorted:
                splits.append(build_object_row(obj, split="skipped", split_reason=reason))
            continue

        # Choose test objects randomly; the rest form the CV pool.
        indices = list(range(num_objects))
        rnd.shuffle(indices)
        test_indices = set(indices[:num_test_objects_per_material])
        cv_indices = set(indices[num_test_objects_per_material:])

        print(f"[INFO] Material {material}: assigning "
              f"{len(test_indices)} objects to TEST and {len(cv_indices)} to CV pool.")

        for i, obj in enumerate(obj_list_sorted):
            if i in test_indices:
                row = build_object_row(obj, split="test", split_reason="test_object")
            else:
                row = build_object_row(obj, split="cv", split_reason="cv_pool")
            splits.append(row)

    # Write object_split.csv
    object_split_csv = output_root / "object_split.csv"
    fieldnames = [
        "material",
        "object_id",
        "split",
        "split_reason",
        "num_images",
        "num_regions",
        "metadata_conflict",
    ]
    with object_split_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(splits)

    print(f"Wrote object-level split to {object_split_csv}")


def write_metadata_aggregated(objects, output_root: Path = OUTPUT_ROOT):
    """
    Write metadata_aggregated.csv with one row per object and per-class
    aggregated types/colors.

    Columns:
      - material
      - object_id
      - body_types
      - body_colors
      - label_types
      - label_colors
      - cap_types
      - cap_colors
      - num_regions
      - metadata_conflict

    metadata_conflict: see compute_metadata_conflict.
    """
    # Class labels to aggregate
    label_names = ["body", "label", "cap"]

    rows = []
    for (material, object_id), obj in sorted(objects.items()):
        per_persp_meta = obj["per_perspective_metadata"]
        num_regions = obj["num_regions"]
        conflict = compute_metadata_conflict(per_persp_meta)

        # Union of all metadata tuples across perspectives
        all_sets = list(per_persp_meta.values())
        union_all = set().union(*all_sets) if all_sets else set()

        # Initialize per-class aggregation
        agg = {
            "body_types": set(),
            "body_colors": set(),
            "label_types": set(),
            "label_colors": set(),
            "cap_types": set(),
            "cap_colors": set(),
        }

        for (label, type_val, color_val) in union_all:
            if label not in label_names:
                continue

            # Types
            if type_val:
                key_types = f"{label}_types"
                agg[key_types].add(type_val)

            # Colors: flatten comma-separated list
            if color_val:
                key_colors = f"{label}_colors"
                for tok in color_val.split(","):
                    tok = tok.strip()
                    if tok:
                        agg[key_colors].add(tok)

        # Convert sets to sorted, comma-separated strings
        row = {
            "material": material,
            "object_id": object_id,
            "body_types": ",".join(sorted(agg["body_types"])) if agg["body_types"] else "",
            "body_colors": ",".join(sorted(agg["body_colors"])) if agg["body_colors"] else "",
            "label_types": ",".join(sorted(agg["label_types"])) if agg["label_types"] else "",
            "label_colors": ",".join(sorted(agg["label_colors"])) if agg["label_colors"] else "",
            "cap_types": ",".join(sorted(agg["cap_types"])) if agg["cap_types"] else "",
            "cap_colors": ",".join(sorted(agg["cap_colors"])) if agg["cap_colors"] else "",
            "num_regions": num_regions,
            "metadata_conflict": conflict,
        }
        rows.append(row)

    metadata_agg_csv = output_root / "metadata_aggregated.csv"
    fieldnames = [
        "material",
        "object_id",
        "body_types",
        "body_colors",
        "label_types",
        "label_colors",
        "cap_types",
        "cap_colors",
        "num_regions",
        "metadata_conflict",
    ]
    with metadata_agg_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote aggregated metadata to {metadata_agg_csv}")

In [ ]:
# === Cell 9: Run Export and Split ===
# Exports LS tasks and writes the YOLO dataset, object_split.csv and
# metadata_aggregated.csv.

client = get_ls_client()
project = client.projects.get(LS_PROJECT_ID)

proj_title = getattr(project, "title", None)
proj_id = getattr(project, "id", None)
print("Project loaded:", proj_title or proj_id)

tasks = export_tasks(client)

processed_images, metadata_rows, total_skipped_small = build_yolo_dataset_flat(
    tasks,
    image_root=IMAGE_ROOT,
    output_root=OUTPUT_ROOT,
    class_map=CLASS_MAP,
)

print(f"Processed images: {len(processed_images)}")
print(f"Polygons skipped due to min_area={POLYGON_MIN_AREA}: {total_skipped_small}")

objects = build_object_index(processed_images, metadata_rows)

aggregate_object_metadata_and_split(objects, output_root=OUTPUT_ROOT)
write_metadata_aggregated(objects, output_root=OUTPUT_ROOT)

In [ ]:
# === Cell 10: Split Summary per Material ===

from collections import defaultdict

def summarize_object_split(output_root: Path = OUTPUT_ROOT):
    object_split_csv = output_root / "object_split.csv"
    if not object_split_csv.exists():
        print(f"object_split.csv not found at {object_split_csv}. Run Cell 9 first.")
        return

    # Load rows
    rows = []
    with object_split_csv.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        print("object_split.csv is empty.")
        return

    # Aggregate stats per material
    stats = defaultdict(lambda: {
        "total": 0,
        "cv": 0,
        "test": 0,
        "skipped": 0,
        "conflict": 0,
    })

    for r in rows:
        material = r["material"]
        split = r["split"]
        conflict = int(r.get("metadata_conflict", 0))

        stats[material]["total"] += 1
        if split in stats[material]:
            stats[material][split] += 1
        else:
            # Count split labels other than cv/test/skipped as well.
            stats[material][split] = stats[material].get(split, 0) + 1

        if conflict:
            stats[material]["conflict"] += 1

    # Print summary
    print(f"Summary of object splits in {object_split_csv}:")
    for material in sorted(stats.keys()):
        s = stats[material]
        print(
            f"  Material {material}: "
            f"total={s['total']}, cv={s['cv']}, test={s['test']}, skipped={s['skipped']}, "
            f"metadata_conflict={s['conflict']}"
        )

# Run summary
summarize_object_split(OUTPUT_ROOT)

In [ ]:
# === Cell 11: Type Conflict Report Helper ===

def _type_conflict_for_meta(per_persp_meta: dict):
    """
    Check type-only conflicts for one object's per_perspective_metadata.

    A label is in conflict if its regions carry more than one distinct non-empty type.

    Returns:
      (overall_conflict_flag: bool, conflict_labels: list[str], type_values_by_label: dict)
    """
    if not per_persp_meta:
        return False, [], {}

    # Flatten all metadata tuples across perspectives into a single set
    all_sets = list(per_persp_meta.values())
    union_all = set().union(*all_sets) if all_sets else set()

    type_values_by_label = {}

    for (label, type_val, _color_val) in union_all:
        if not label:
            continue
        normalized_type = (type_val or "").strip()
        if normalized_type:
            type_values_by_label.setdefault(label, set()).add(normalized_type)

    conflict_labels = []
    for label, type_vals in type_values_by_label.items():
        if len(type_vals) > 1:
            conflict_labels.append(label)

    overall_conflict = bool(conflict_labels)
    return overall_conflict, sorted(conflict_labels), type_values_by_label


def summarize_type_conflicts(objects_dict=None, only_conflicts=True):
    """
    Run the type-only consistency check over all objects and print a report.

    Parameters
    ----------
    objects_dict : dict or None
        If None, uses the global 'objects' built in Cell 9.
    only_conflicts : bool
        If True (default), only prints objects with TYPE conflicts.
        If False, prints a line for every object.

    Output format (per object)
    --------------------------
      [TYPE] HDPE obj 01: regions=23, type_conflict=YES, labels=body,cap
    """
    if objects_dict is None:
        if "objects" not in globals():
            print("Error: 'objects' dict not found. Run Cell 9 first.")
            return
        objects_dict = objects

    total_objects = 0
    total_conflicts = 0

    print("=== TYPE-only metadata consistency report ===")
    for (material, object_id), obj in sorted(objects_dict.items()):
        total_objects += 1
        per_persp_meta = obj["per_perspective_metadata"]
        num_regions = obj["num_regions"]

        conflict, conflict_labels, type_values_by_label = _type_conflict_for_meta(per_persp_meta)

        if conflict:
            total_conflicts += 1

        if only_conflicts and not conflict:
            continue

        labels_str = ",".join(conflict_labels) if conflict_labels else "-"
        status = "YES" if conflict else "NO"
        print(f"[TYPE] {material} obj {object_id}: regions={num_regions}, "
              f"type_conflict={status}, labels={labels_str}")

    print("\nSummary:")
    print(f"  total objects      = {total_objects}")
    print(f"  objects with conflict = {total_conflicts}")
    if total_objects:
        print(f"  conflict rate      = {total_conflicts}/{total_objects} "
              f"({100.0 * total_conflicts / total_objects:.1f}%)")
    else:
        print("  (no objects found)")

# Example usage:
# summarize_type_conflicts()                 # show only objects with conflicts
# summarize_type_conflicts(only_conflicts=False)  # show all objects

In [ ]:
# === Cell 12: Run Type Conflict Report ===
summarize_type_conflicts()

In [ ]:
# === Cell 13: Visualization Setup ===

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import cv2
from pathlib import Path

# Inverse class map: id -> name (from CLASS_MAP in Cell 4)
CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

# Per-class BGR colors for OpenCV
CLASS_COLOR_BRG = {
    "body":  (0,   0, 255),  # red
    "label": (255, 0,   0),  # blue
    "cap":   (0, 255,   0),  # green
}

# Fallback color for unknown classes
DEFAULT_COLOR_BRG = (0, 255, 255)  # yellow


def parse_yolo_segmentation_label(label_path: Path):
    """
    Parse a YOLOv8 segmentation label file.

    Returns a list of dicts:
      - class_id (int)
      - class_name (str or None)
      - points (list of (x_norm, y_norm) float pairs)
    """
    objects = []
    if not label_path.exists():
        return objects

    with label_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 7:  # cls + at least 3 points (6 coords)
                continue

            try:
                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))
            except ValueError:
                continue

            if len(coords) % 2 != 0:
                # Skip malformed lines.
                continue

            points = []
            for i in range(0, len(coords), 2):
                x = coords[i]
                y = coords[i + 1]
                points.append((x, y))

            class_name = CLASS_ID_TO_NAME.get(cls_id)
            objects.append({
                "class_id": cls_id,
                "class_name": class_name,
                "points": points,
            })

    return objects


# Build LS-based image_index for brush visualization
image_index = []

if 'tasks' not in globals():
    print("Variable 'tasks' not defined. Run Cell 9 (export) first.")
else:
    img_key = detect_image_key(tasks)
    for t_idx, t in enumerate(tasks):
        data = t.get("data", {})
        if img_key not in data:
            continue

        image_value = data[img_key]
        image_path = resolve_image_path(image_value)

        # Skip images missing on disk
        if not image_path.exists():
            continue

        annotations = t.get("annotations") or []
        if not annotations:
            continue

        ann = annotations[0]  # one annotation per image
        results = ann.get("result") or []

        # Collect all brushlabels regions
        brush_results = [
            r for r in results
            if r.get("type") == "brushlabels"
        ]

        if not brush_results:
            continue

        image_index.append({
            "task_index": t_idx,
            "task_id": t.get("id"),
            "image_path": image_path,
            "image_name": image_path.name,
            "brush_results": brush_results,
        })

    print(f"Indexed {len(image_index)} images with brushlabels regions from LS.")
    if image_index:
        print("Example:", image_index[0]["image_name"], "with",
              len(image_index[0]["brush_results"]), "brush regions")

In [ ]:
# === Cell 14: LS Mask vs YOLO Plot ===

def visualize_ls_mask_with_yolo(image_path: Path, brush_results, alpha=0.4, save_path=None):
    """
    Plot three panels: exported image, LS brush-mask overlay, and YOLO polygons
    from OUTPUT_ROOT/labels/all/<stem>.txt (colors from CLASS_COLOR_BRG).

    If save_path is given, the figure is also saved as a PNG.
    """
    # Prefer the exported image from images/all
    exported_img_path = OUTPUT_ROOT / "images" / "all" / image_path.name
    if exported_img_path.exists():
        img_path_to_use = exported_img_path
    else:
        img_path_to_use = image_path

    # Read image
    img_bgr = cv2.imread(str(img_path_to_use))
    if img_bgr is None:
        print("Could not read image:", img_path_to_use)
        return

    h, w = img_bgr.shape[:2]

    # 1) Exported image
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2) LS mask overlay (per-class colors)
    blended_ls = img_bgr.astype(float)
    for r in brush_results:
        mask = brush_result_to_mask(r)
        if mask.shape[:2] != (h, w):
            print(f"WARNING: LS mask shape {mask.shape} does not match image shape {(h, w)}")
            continue

        mask_bool = mask.astype(bool)

        label_name = r["value"].get("brushlabels", ["?"])[0]
        color_bgr = CLASS_COLOR_BRG.get(label_name, DEFAULT_COLOR_BRG)
        color_arr = np.array(color_bgr, dtype=float)

        blended_ls[mask_bool] = (1 - alpha) * blended_ls[mask_bool] + alpha * color_arr

    blended_ls = blended_ls.astype(np.uint8)
    blended_ls_rgb = cv2.cvtColor(blended_ls, cv2.COLOR_BGR2RGB)

    # 3) YOLO polygon outlines on the plain image
    yolo_img_bgr = img_bgr.copy()

    lbl_all_dir = OUTPUT_ROOT / "labels" / "all"
    label_path = lbl_all_dir / f"{image_path.stem}.txt"

    yolo_objs = parse_yolo_segmentation_label(label_path) if label_path.exists() else []
    debug_lines = []

    if not label_path.exists():
        print("NO YOLO label file found for:", label_path)
    elif not yolo_objs:
        print("YOLO label file exists but contains no valid polygons:", label_path)
    else:
        for obj in yolo_objs:
            cls_id = obj["class_id"]
            class_name = obj["class_name"]
            pts_norm = obj["points"]

            color_bgr = CLASS_COLOR_BRG.get(class_name, DEFAULT_COLOR_BRG)
            debug_lines.append(f"class_id={cls_id}, class_name={class_name}, points={len(pts_norm)}")

            # Denormalize polygon points
            poly = []
            for (x_norm, y_norm) in pts_norm:
                x_px = int(round(x_norm * w))
                y_px = int(round(y_norm * h))
                poly.append([x_px, y_px])
            poly = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)

            # Draw polygon outline only
            cv2.polylines(yolo_img_bgr, [poly], isClosed=True, color=color_bgr, thickness=2)

    yolo_rgb = cv2.cvtColor(yolo_img_bgr, cv2.COLOR_BGR2RGB)

    # Plot three panels
    fig = plt.figure(figsize=(16, 6))

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_title("Exported Image")
    ax1.imshow(img_rgb)
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_title("Label-Studio Masks")
    ax2.imshow(blended_ls_rgb)
    ax2.axis("off")

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_title("Exported YOLO Polygons")
    ax3.imshow(yolo_rgb)
    ax3.axis("off")

    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved visualization to: {save_path}")

    plt.show()
    plt.close(fig)

    if yolo_objs:
        print("YOLO objects:")
        for line in debug_lines:
            print("  ", line)
    else:
        print("No YOLO polygons rendered for this image.")

In [ ]:
# === Cell 15: Interactive LS/YOLO Viewer ===

if not image_index:
    print("No LS images with brushlabels in image_index. Run Cell 13 first.")
else:
    # Dropdown for image selection (by LS image_index)
    ls_yolo_options = [
        (f"{i}: {entry['image_name']} (task {entry['task_id']})", i)
        for i, entry in enumerate(image_index)
    ]
    ls_yolo_image_dropdown = widgets.Dropdown(
        options=ls_yolo_options,
        description="Image:",
        layout=widgets.Layout(width="600px")
    )

    # Buttons: show, random image, save PNG
    ls_yolo_button = widgets.Button(
        description="Show",
        button_style="primary"
    )

    ls_yolo_random_button = widgets.Button(
        description="Random",
        button_style="info",
        icon="random"
    )

    ls_yolo_save_button = widgets.Button(
        description="Save PNG",
        button_style="",
        icon="save"
    )

    # Output area
    ls_yolo_output = widgets.Output()

    def _show_ls_yolo_image(idx=None):
        if idx is None:
            idx = ls_yolo_image_dropdown.value
        entry = image_index[idx]
        image_path = entry["image_path"]
        brush_results = entry["brush_results"]
        visualize_ls_mask_with_yolo(image_path, brush_results)

    def on_ls_yolo_clicked(b):
        with ls_yolo_output:
            ls_yolo_output.clear_output()
            _show_ls_yolo_image()

    def on_ls_yolo_random_clicked(b):
        with ls_yolo_output:
            ls_yolo_output.clear_output()
            new_idx = random.randrange(len(image_index))
            ls_yolo_image_dropdown.value = new_idx
            _show_ls_yolo_image(idx=new_idx)

    def on_ls_yolo_save_clicked(b):
        with ls_yolo_output:
            ls_yolo_output.clear_output()
            idx = ls_yolo_image_dropdown.value
            entry = image_index[idx]
            image_path = entry["image_path"]
            brush_results = entry["brush_results"]

            out_dir = OUTPUT_ROOT / "results"
            out_dir.mkdir(parents=True, exist_ok=True)

            img_stem = Path(image_path).stem
            filename = f"{OUTPUT_ROOT.name}_lsviz_{img_stem}.png"
            save_path = out_dir / filename

            print(f"Saving visualization to: {save_path}")
            visualize_ls_mask_with_yolo(image_path, brush_results, save_path=save_path)

    ls_yolo_button.on_click(on_ls_yolo_clicked)
    ls_yolo_random_button.on_click(on_ls_yolo_random_clicked)
    ls_yolo_save_button.on_click(on_ls_yolo_save_clicked)

    # Display controls and output
    ls_yolo_controls = widgets.VBox([
        ls_yolo_image_dropdown,
        widgets.HBox([ls_yolo_button, ls_yolo_random_button, ls_yolo_save_button])
    ])
    display(ls_yolo_controls, ls_yolo_output)